In [ ]:
import sys

from pathlib import Path

module_path = str(Path("..").resolve())
if module_path not in sys.path:
    sys.path.append(module_path)

from config_loaders.dataset_config_loader import DatasetConfig
from datasets import load_dataset
from utils.data_io import make_dir, load_jsonl, save_jsonl
from utils.question_formatter import QuestionFormatter

In [2]:
dataset = "llava-bench"
ds_config = DatasetConfig()
ds_paths = ds_config.get_ds_paths(dataset)
ds_paths

{'main': {'imgs': 'data/datasets/llava-bench/imgs/main',
  'questions': 'data/datasets/llava-bench/q.jsonl'}}

In [3]:
question_formatter = QuestionFormatter(dataset)

In [4]:
ds = load_dataset("liuhaotian/llava-bench-in-the-wild")

Resolving data files:   0%|          | 0/29 [00:00<?, ?it/s]

In [5]:
ds.keys()

dict_keys(['train'])

Save images

In [6]:
for q_class in ds_paths:
    out_dir = f"../{ds_paths[q_class]['imgs']}"
    make_dir(out_dir)

    i = 1
    for entry in ds["train"]:
        img = entry["image"]
        rgb_img = img.convert("RGB")
        rgb_img.save(f"{out_dir}/{str(i).zfill(3)}.jpg")
        i += 1

[Errno 17] File exists: '../data/datasets/llava-bench/imgs/main'


Format and save questions

Questions for LLaVa-Bench do not come with the images.
They are first manually downloaded from [Hugging Face](https://huggingface.co/datasets/liuhaotian/llava-bench-in-the-wild).

In [ ]:
questions = load_jsonl(f"../{ds_paths['main']['questions']}")
new_q = [question_formatter.format(raw_q, "main") for raw_q in questions]

In [8]:
save_jsonl(new_q, f"../{ds_paths['main']['questions']}")